# Introduction
This notebook is a clone of the run_quantize.py file for mimicking and debugging experiments without using seml and slurm

# 1. Initial Setup

In [1]:
print("Hello World")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

Hello World
Free GPU Memory (GB): 39.3896


In [2]:
# Setting up environment
print("\n################################")
print("Setting up environment...")
print("################################\n")

import os
#os.chdir('..')
print("Current Working Directory ", os.getcwd())
import sys
sys.path.append("../") # Add directory containing src/data to path

import importlib
import src  # Assuming src is the package name

# Reload the src module after making changes
importlib.reload(src)

%load_ext autoreload
%autoreload 2

import seml
import re
import shutil

os.environ["TOKENIZERS_PARALLELISM"] = "false"  # Disables parallelism to remove transformers warning

print("\n################################")
print("Setting up cache paths...")
print("################################\n")

os.environ["MKL_SERVICE_FORCE_INTEL"] = "1"
CACHE_PATH = "/nfs/students/daro/.cache/huggingface"
HUB_PATH = "/nfs/students/daro/.cache/huggingface/hub/"

if not os.path.exists(HUB_PATH):
    os.makedirs(HUB_PATH)
    print(f"Creating huggingface hub path at {HUB_PATH}")
    
print(f"Setting cache path to {CACHE_PATH}")
os.environ["TORCH_HOME"] = CACHE_PATH
os.environ["HF_HOME"] = CACHE_PATH

import torch
torch.hub.set_dir(CACHE_PATH)
with torch.no_grad():
    torch.cuda.empty_cache()
    
import logging
logger = logging.getLogger("quant_logger")
    
!cat /proc/meminfo | awk '/MemTotal/ {total=$2} /MemFree/ {free=$2} /MemAvailable/ {available=$2} END {printf "MemTotal: %.2f GB\nMemFree: %.2f GB\nMemAvailable: %.2f GB\n", total/1024/1024, free/1024/1024, available/1024/1024}'
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

print("\n################################")
print("Setting up cuda devices...")
print("################################\n")

if torch.cuda.is_available():
    print("CUDA device is available!")
    # Get the number of available CUDA devices
    num_cuda_devices = torch.cuda.device_count()
    print(f"Number of CUDA devices: {num_cuda_devices}")
    
    # Loop through available devices and get name
    for device_id in range(num_cuda_devices):
        device = torch.device(f"cuda:{device_id}")
        name = torch.cuda.get_device_name(device)
        print(f"  - CUDA Device {device_id+1}: {name}")
else:
    print("CUDA device is not available.")
    
print("\n################################")
print("Authentication with Hugging Face...")
print("################################\n")

import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()
huggingface_token = os.getenv('HUGGINGFACE_TOKEN')

if huggingface_token is None:
    raise ValueError("Please set the HUGGINGFACE_TOKEN environment variable.")
else:
    print("Hugging Face token loaded successfully.")

login(token=huggingface_token, add_to_git_credential=True)
print("Successfully authenticated with the Hugging Face API.")

print("\n################################")
print("Setting up GPU memory usage list...")
print("################################\n")
# Global list to store GPU memory usage
from src.evaluations.evaluate_memory import record_gpu_memory
gpu_memory_usage = {}
record_gpu_memory(gpu_memory_usage=gpu_memory_usage, context="Warm up notebook")


################################
Setting up environment...
################################

Current Working Directory  /nfs/homedirs/daro/git/quantization-reliability

################################
Setting up cache paths...
################################

Setting cache path to /nfs/students/daro/.cache/huggingface
MemTotal: 1007.72 GB
MemFree: 149.10 GB
MemAvailable: 636.33 GB
Free GPU Memory (GB): 39.3896

################################
Setting up cuda devices...
################################

CUDA device is available!
Number of CUDA devices: 1
  - CUDA Device 1: NVIDIA A100-PCIE-40GB

################################
Authentication with Hugging Face...
################################

Hugging Face token loaded successfully.
Token is valid (permission: write).
Your token has been saved in your configured git credential helpers (store).
Your token has been saved to /nfs/students/daro/.cache/huggingface/token
Login successful
Successfully authenticated with the Hugging Face

# 2. SEML Pipeline

# 2.1 Response Generator

In [4]:
exp_id = "09-17-1-test"

# 2.2 Pipeline

# 3. Load Quantize

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from urllib.error import HTTPError
from typing import Tuple, Optional

import os
import torch
from typing import Tuple, Optional
import logging

from hqq.engine.hf import HQQModelForCausalLM
from hqq.models.hf.base import AutoHQQHFModel
from awq import AutoAWQForCausalLM

from src import MODEL_SAVE_PATH
logger = logging.getLogger("quant_logger")

# Base models dictionary
base_models = {
    # TinyLlama models
    "TinyLlama-Chat": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "TinyLlama": "TinyLlama/TinyLlama_v1.1",
    
    # Meta models
    "Llama-3-8B": "meta-llama/Meta-Llama-3-8B",
    
    # BigScience models
    "Bloomz": "bigscience/bloomz-1b1",
    
    # OpenAI community models
    "GPT2-Large": "openai-community/gpt2-large",
}

# Hugging Face quantized models dictionary
hf_quantized_models = {
    # AQLM quantized models
    "Llama-3-8B-AQLM-2Bit": "ISTA-DASLab/Meta-Llama-3-8B-AQLM-2Bit-1x16",
    "Llama-3-8B-AQLM-PV-2Bit": "ISTA-DASLab/Meta-Llama-3-8B-AQLM-PV-2Bit-1x16",
    "Llama-3-8B-AQLM-PV-1Bit": "ISTA-DASLab/Meta-Llama-3-8B-AQLM-PV-1Bit-1x16",
    
    # AWQ quantized models
    "Llama-3-8B-AWQ-4bit": "PrunaAI/meta-llama-Meta-Llama-3-8B-AWQ-4bit-smashed",
    
    # BitsAndBytes (BNB) quantized models
    "Llama-3-8B-16K-bnb-4bit": "PrunaAI/mattshumer-Llama-3-8B-16K-bnb-4bit-smashed",
    
    # HQQ quantized models
    "Llama-3-8B-HQQ-4bit": "PrunaAI/meta-llama-Meta-Llama-3-8B-HQQ-4bit-smashed",
    "Llama-3-8B-HQQ-2bit": "PrunaAI/meta-llama-Meta-Llama-3-8B-HQQ-2bit-smashed",
    "Llama-3-8B-HQQ-1bit": "PrunaAI/meta-llama-Meta-Llama-3-8B-HQQ-1bit-smashed",
}

local_quantized_models = {
    # AWQ models
    "Llama-3-8B-AWQ-4bit-local": os.path.join(MODEL_SAVE_PATH, "Meta-Llama-3-8B-AWQ-4"),
    
    # BNB models
    "Llama-3-8B-BNB-8bit-local": os.path.join(MODEL_SAVE_PATH, "Meta-Llama-3-8B-BNB-8"),
    "Llama-3-8B-BNB-4bit-local": os.path.join(MODEL_SAVE_PATH, "Meta-Llama-3-8B-BNB-4"),
    
    # HQQ models
    "Llama-3-8B-HQQ-8-uniform-local": os.path.join(MODEL_SAVE_PATH, "Meta-Llama-3-8B-HQQ-8-uniform"),
    "Llama-3-8B-HQQ-mixed-local": os.path.join(MODEL_SAVE_PATH, "Meta-Llama-3-8B-HQQ-mixed"),
    
    # QUANTO models
    "Llama-3-8B-QUANTO-local": os.path.join(MODEL_SAVE_PATH, "Meta-Llama-3-8B-QUANTO"),
    "Llama-3-8B-QUANTO-CALIB-local": os.path.join(MODEL_SAVE_PATH, "Meta-Llama-3-8B-QUANTO-CALIB"),
    "Llama-3-8B-QUANTO-QAT-local": os.path.join(MODEL_SAVE_PATH, "Meta-Llama-3-8B-QUANTO-QAT"),
    
    # HQQ-LORA models
    "Llama-3-8B-HQQ-LORA-local": os.path.join(MODEL_SAVE_PATH, "Meta-Llama-3-8B-HQQ-LORA"),
    
    # AQLM-LORA models
    "Llama-3-8B-AQLM-LORA-local": os.path.join(MODEL_SAVE_PATH, "Meta-Llama-3-8B-AQLM-LORA"),
}

META_LLAMA_3_8B = "meta-llama/Meta-Llama-3-8B"
local_tokenizers = {
    "Llama-3-8B-AWQ-4bit-local": META_LLAMA_3_8B,
    "Llama-3-8B-BNB-8bit-local": META_LLAMA_3_8B,
    "Llama-3-8B-BNB-4bit-local": META_LLAMA_3_8B,
    "Llama-3-8B-HQQ-8-uniform-local": META_LLAMA_3_8B,
    "Llama-3-8B-HQQ-mixed-local": META_LLAMA_3_8B,
    "Llama-3-8B-QUANTO-local": META_LLAMA_3_8B,
    "Llama-3-8B-QUANTO-CALIB-local": META_LLAMA_3_8B,
    "Llama-3-8B-QUANTO-QAT-local": META_LLAMA_3_8B,
    "Llama-3-8B-HQQ-LORA-local": META_LLAMA_3_8B,
    "Llama-3-8B-AQLM-LORA-local": META_LLAMA_3_8B,
}


from src import MODEL_SAVE_PATH

def load_model_and_tokenizer(
    model_name: str,
    device: str = "cuda",
    max_memory: Optional[dict] = None,
    cache_dir: Optional[str] = None
) -> Tuple[AutoModelForCausalLM, AutoTokenizer]:
    """
    Load a model and tokenizer from Hugging Face or a locally quantized model.

    Args:
        model_name (str): Short name of the model to load.
        device (str): Device to load the model on. Defaults to "cuda".
        max_memory (dict, optional): Maximum memory to use for model loading.

    Returns:
        Tuple[AutoModelForCausalLM, AutoTokenizer]: Loaded model and tokenizer.

    Raises:
        ValueError: If the model name is not found or the local model is invalid.
        OSError: If there's an error loading the model or tokenizer.
    """
    try:
        logger.info(f"Attempting to load model: {model_name}")
        
        # Determine the actual model path
        if model_name in local_quantized_models:
            model_path = local_quantized_models[model_name]
            is_local_model = True
        elif model_name in hf_quantized_models:
            model_path = hf_quantized_models[model_name]
            is_local_model = False
        elif model_name in base_models:
            model_path = base_models[model_name]
            is_local_model = False
        else:
            raise ValueError(f"Model {model_name} not found in any of the model dictionaries")

        if is_local_model:
            logger.info(f"Loading locally quantized model from: {model_path}")
            if not os.path.exists(model_path):
                raise OSError(f"Local model path does not exist: {model_path}")

        # Special handling for HQQ models
        if "HQQ" in model_name:
            try:
                model = HQQModelForCausalLM.from_quantized(model_path, device_map='auto', cache_dir=cache_dir)
            except:
                model = AutoHQQHFModel.from_quantized(model_path, device_map='auto', cache_dir=cache_dir)
            tokenizer = AutoTokenizer.from_pretrained(local_tokenizers[model_name] if is_local_model else model_path, device_map='auto', cache_dir=cache_dir)

        # Special handling for AWQ model from PrunaAI
        elif "AWQ" in model_name:
            model = AutoAWQForCausalLM.from_pretrained(model_path, trust_remote_code=True, torch_dtype=torch.float16, device_map='auto', cache_dir=cache_dir)
            tokenizer = AutoTokenizer.from_pretrained(local_tokenizers[model_name] if is_local_model else model_path, device_map='auto', cache_dir=cache_dir)
            model.dtype = torch.float16
        else:
            # Load from Hugging Face or local path
            model = AutoModelForCausalLM.from_pretrained(
                model_path,
                torch_dtype="auto",
                device_map=device,
                max_memory=max_memory,
                cache_dir=cache_dir
            )
            tokenizer = AutoTokenizer.from_pretrained(local_tokenizers[model_name] if is_local_model else model_path, device_map='auto', cache_dir=cache_dir)
        
        model.NAME = model_name
        tokenizer.pad_token_id = tokenizer.eos_token_id
        tokenizer.padding_side = "left"
        
        if tokenizer.model_max_length > 1e6:
            logger.warning(f"Tokenizer model max length reduced from {tokenizer.model_max_length} to 2048 to fit in memory")
            tokenizer.model_max_length = 2048
        
        logger.info(f"Successfully loaded model: {model_name}")
        logger.info(f"Model configuration:")
        logger.info(f"Model cache directory: {getattr(model, 'cache_dir', 'Not available')}")
        logger.info(f"- Model max length: {tokenizer.model_max_length}")
        logger.info(f"- Model dtype: {getattr(model, 'dtype', 'Not available')}")
        logger.info(f"- Model device: {getattr(model, 'device', 'Not available')}")
        logger.info(f"- Model parameters: {getattr(model, 'num_parameters', 'Not available')}")
        logger.info(f"- Model memory footprint: {getattr(model, 'memory_footprint', 0) / (1024 ** 3):.2f} GB")
        logger.info(f"- Vocabulary size: {tokenizer.vocab_size}")
        logger.info(f"- Padding token ID: {tokenizer.pad_token_id}")
        logger.info(f"- Special tokens: {tokenizer.special_tokens_map}")
        
        return model, tokenizer
    
    except ValueError as ve:
        logger.error(f"ValueError: {ve}")
        raise
    except OSError as ose:
        logger.error(f"OSError: {ose}")
        raise
    except Exception as e:
        logger.error(f"Unexpected error occurred while loading model: {e}")
        raise

def get_model(model_name=None, directory_model=None, seed=123, device="cuda"):
    logger.info(f"Loading model {model_name}")
    torch.manual_seed(seed)
    if model_name is not None:
        model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map=device)
        model.NAME = model_name
    elif model_name is None and directory_model is not None:
        # Load model from local directory
        try:
            model = AutoModelForCausalLM.from_pretrained(directory_model)
            model.NAME = model_name
        except (OSError, HTTPError) as e:
            print(f"Error loading model from directory: {directory_model}")
            print(f"Error message: {e}")
    else:
        # No model name or directory provided, raise an error
        raise ValueError("Please specify either model_name or directory_model")
    
    return model


def get_tokenizer(model_name=None, directory_model=None, cache_dir=None, seed=123, device="cuda"):
    logger.info(f"Loading tokenizer {model_name}")
    torch.manual_seed(seed)
    if model_name is not None:
        tokenizer = AutoTokenizer.from_pretrained(model_name, device_map=device, trust_remote_code=True)
    elif model_name is None and directory_model is not None:
        # Load tokenizer from local directory
        try:
            tokenizer = AutoTokenizer.from_pretrained(directory_model, trust_remote_code=True)
        except (OSError, HTTPError) as e:
            print(f"Error loading tokenizer from directory: {directory_model}")
            print(f"Error message: {e}")
    else:
        # No model name or directory provided, raise an error
        raise ValueError("Please specify either model_name or directory_model")
    
    return tokenizer


def get_model_name(model_name):
    try:
        return base_models[model_name]
    except KeyError:
        raise NotImplementedError(f"Model {model_name} is not spelled correctly or not yet supported")

2024-09-27 12:31:20.694081: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-09-27 12:31:20.715518: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-09-27 12:31:20.722036: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-09-27 12:31:20.738077: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-09-27 12:31:23.290316: W tensorflow/compiler/tf2

## 3.1 Test Quantize

In [4]:
import torch

def test_model_loading(model_name_or_path, device="cuda", max_memory=None):
    print(f"\nTesting model: {model_name}")
    print("-" * 50)
    try:
        model, tokenizer = load_model_and_tokenizer(model_name_or_path, device=device, max_memory=max_memory)
        print(f"Successfully loaded model: {model_name}")
        print(f"Model configuration:")
        print(f"- Model max length: {tokenizer.model_max_length}")
        print(f"- Model dtype: {getattr(model, 'dtype', 'Not available')}")
        print(f"- Model device: {getattr(model, 'device', 'Not available')}")
        print(f"- Model parameters: {getattr(model, 'num_parameters', lambda: 'Not available')():,}")
        print(f"- Model memory footprint: {getattr(model, 'get_memory_footprint', lambda: 0)() / (1024 ** 3):.2f} GB")
        print(f"- Vocabulary size: {tokenizer.vocab_size}")
        print(f"- Padding token ID: {tokenizer.pad_token_id}")
        print(f"- Special tokens: {tokenizer.special_tokens_map}")
    except Exception as e:
        print(f"Error loading model {model_name}: {str(e)}")
    print("-" * 50)

# Test base models
# print("\n=== Testing Base Models ===")
# for model_name, hf_path in base_models.items():
#     test_model_loading(model_name)

# Test HF quantized models
print("\n=== Testing HF Quantized Models ===")
for model_name, hf_path in hf_quantized_models.items():
    test_model_loading(model_name)

# Test local quantized models
print("\n=== Testing Local Quantized Models ===")
for model_name, local_path in local_quantized_models.items():
    test_model_loading(model_name)

# Test invalid model name
print("\n=== Testing Invalid Model Name ===")
test_model_loading("invalid_model_name")

# Test nonexistent local model
print("\n=== Testing Nonexistent Local Model ===")
test_model_loading("nonexistent_local_model")

# Test with different device
print("\n=== Testing Model Loading on CPU ===")
test_model_loading("meta-llama/Llama-2-7b-hf", device="cpu")

# Test with max memory setting
print("\n=== Testing Model Loading with Max Memory Setting ===")
max_memory = {"cuda:0": "10GiB"}
test_model_loading("meta-llama/Llama-2-7b-hf", max_memory=max_memory)

print("\nAll tests completed.")


=== Testing HF Quantized Models ===

Testing model: Llama-3-8B-AQLM-2Bit
--------------------------------------------------


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Tokenizer model max length reduced from 1000000000000000019884624838656 to 2048 to fit in memory


Successfully loaded model: Llama-3-8B-AQLM-2Bit
Model configuration:
- Model max length: 2048
- Model dtype: torch.float16
- Model device: cuda:0
- Model parameters: 2,042,171,392
- Model memory footprint: 3.80 GB
- Vocabulary size: 128000
- Padding token ID: 128001
- Special tokens: {'bos_token': '<|begin_of_text|>', 'eos_token': '<|end_of_text|>', 'pad_token': '<|end_of_text|>'}
--------------------------------------------------

Testing model: Llama-3-8B-AQLM-PV-2Bit
--------------------------------------------------


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Tokenizer model max length reduced from 1000000000000000019884624838656 to 2048 to fit in memory


Successfully loaded model: Llama-3-8B-AQLM-PV-2Bit
Model configuration:
- Model max length: 2048
- Model dtype: torch.float16
- Model device: cuda:0
- Model parameters: 2,042,171,392
- Model memory footprint: 3.80 GB
- Vocabulary size: 128000
- Padding token ID: 128001
- Special tokens: {'bos_token': '<|begin_of_text|>', 'eos_token': '<|end_of_text|>', 'pad_token': '<|end_of_text|>'}
--------------------------------------------------

Testing model: Llama-3-8B-AQLM-PV-1Bit
--------------------------------------------------


...

Successfully loaded model: Llama-3-8B-HQQ-8-uniform-local
Model configuration:
- Model max length: 2048
- Model dtype: torch.float16
- Model device: cuda:0
- Model parameters: 8,030,261,248
- Model memory footprint: 8.46 GB
- Vocabulary size: 128000
- Padding token ID: 128001
- Special tokens: {'bos_token': '<|begin_of_text|>', 'eos_token': '<|end_of_text|>', 'pad_token': '<|end_of_text|>'}
--------------------------------------------------

Testing model: Llama-3-8B-HQQ-mixed-local
--------------------------------------------------


100%|██████████| 225/225 [00:00<00:00, 6835.47it/s]
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Tokenizer model max length reduced from 1000000000000000019884624838656 to 2048 to fit in memory
OSError: Local model path does not exist: /nfs/students/daro/models/Meta-Llama-3-8B-QUANTO
OSError: Local model path does not exist: /nfs/students/daro/models/Meta-Llama-3-8B-QUANTO-CALIB
OSError: Local model path does not exist: /nfs/students/daro/models/Meta-Llama-3-8B-QUANTO-QAT


Successfully loaded model: Llama-3-8B-HQQ-mixed-local
Model configuration:
- Model max length: 2048
- Model dtype: torch.float16
- Model device: cuda:0
- Model parameters: 2,426,671,104
- Model memory footprint: 5.21 GB
- Vocabulary size: 128000
- Padding token ID: 128001
- Special tokens: {'bos_token': '<|begin_of_text|>', 'eos_token': '<|end_of_text|>', 'pad_token': '<|end_of_text|>'}
--------------------------------------------------

Testing model: Llama-3-8B-QUANTO-local
--------------------------------------------------
Error loading model Llama-3-8B-QUANTO-local: Local model path does not exist: /nfs/students/daro/models/Meta-Llama-3-8B-QUANTO
--------------------------------------------------

Testing model: Llama-3-8B-QUANTO-CALIB-local
--------------------------------------------------
Error loading model Llama-3-8B-QUANTO-CALIB-local: Local model path does not exist: /nfs/students/daro/models/Meta-Llama-3-8B-QUANTO-CALIB
--------------------------------------------------

Te

  0%|          | 0/225 [00:00<?, ?it/s]
Unexpected error occurred while loading model: Cannot copy out of meta tensor; no data! Please use torch.nn.Module.to_empty() instead of torch.nn.Module.to() when moving module from meta to a different device.
OSError: Local model path does not exist: /nfs/students/daro/models/Meta-Llama-3-8B-AQLM-LORA
ValueError: Model invalid_model_name not found in any of the model dictionaries
ValueError: Model nonexistent_local_model not found in any of the model dictionaries
ValueError: Model meta-llama/Llama-2-7b-hf not found in any of the model dictionaries
ValueError: Model meta-llama/Llama-2-7b-hf not found in any of the model dictionaries


Error loading model Llama-3-8B-HQQ-LORA-local: Cannot copy out of meta tensor; no data! Please use torch.nn.Module.to_empty() instead of torch.nn.Module.to() when moving module from meta to a different device.
--------------------------------------------------

Testing model: Llama-3-8B-AQLM-LORA-local
--------------------------------------------------
Error loading model Llama-3-8B-AQLM-LORA-local: Local model path does not exist: /nfs/students/daro/models/Meta-Llama-3-8B-AQLM-LORA
--------------------------------------------------

=== Testing Invalid Model Name ===

Testing model: Llama-3-8B-AQLM-LORA-local
--------------------------------------------------
Error loading model Llama-3-8B-AQLM-LORA-local: Model invalid_model_name not found in any of the model dictionaries
--------------------------------------------------

=== Testing Nonexistent Local Model ===

Testing model: Llama-3-8B-AQLM-LORA-local
--------------------------------------------------
Error loading model Llama-3-8